validation using Strategy pattern

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, List, Union, Optional, Any
from loguru import logger
import pandas as pd
import numpy as np

class ValidationStrategy(ABC):
    def __init__(self, config: Dict):
        # config is a dictionary that contains the configuration for the strategy
        self.config = config

    @abstractmethod
    def validate(self, data) -> pd.Series:
        '''Return vectorized validation result. True for invalid, False for valid.'''
        pass

    def notna(self, data) -> pd.Series:
        """Return vectorized validation result. True for not na, False for na."""
        return data.notna()

    def is_empty(self, data) -> bool:
        """Check if data is empty"""
        if data is None or data.empty:
            raise ValueError(self.message)
        return False

    def make_result(self, data):
        pass


In [ ]:
class MandatoryValidation(ValidationStrategy):

    def __init__(self, config: Dict, empty_list: Optional[List] = None):
        super().__init__(config)
        self.empty_list = empty_list or []

    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        mask: pd.Series = data.isna()
        if self.empty_list:
            mask = mask | (data.isin(self.empty_list))
        return mask

In [ ]:
master_data_setup_file_path = "/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/prevalidation_data_2503/Data Template 03_Master Data Setup_SLP3P4 1.xlsx"
df_internal_suspended_list: pd.DataFrame = pd.read_excel(master_data_setup_file_path, sheet_name="InternalSuspendedList")


In [ ]:
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]

In [ ]:
mandatory_columns = ["ID", "Debtor type"]
for column in mandatory_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip()
    # Validation
    validation_result = MandatoryValidation(config={},empty_list=["", "nan"]).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is mandatory"}))
    # Log result
    logger.info(f"[{column}] [Check mandatory] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid, Excel index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
df_internal_suspended_list

In [ ]:
f"{pd.Series( [True, False, True]).sum()}"

In [ ]:
class UniqueValidation(ValidationStrategy):
    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        mask: pd.Series = data.duplicated(keep=False) & data.notna()
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
unique_columns = ["ID"]
for column in unique_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    validation_result = UniqueValidation(config={}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is unique"}))
    # Log result
    logger.info(f"[{column}] [Check unique] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Excel index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
df_internal_suspended_list

In [ ]:
class ValueListValidation(ValidationStrategy):
    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        mask: pd.Series = ~data.isin(self.config["value_list"]) & data.notna()
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
value_list_columns = ["Debtor type"]
for column in value_list_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().str.title().replace("", np.nan)
    # Validation
    validation_result = ValueListValidation({"value_list": ["Individual", "Company"]}).validate(df_internal_suspended_list[column])
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not in value list"}))
    # Log result
    logger.info(f"[{column}] [Check value list] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
df_internal_suspended_list

In [ ]:
# # Error now
# class RegexValidation(ValidationStrategy):
#     def validate(self, data) -> pd.Series:
#         self.is_empty(data)
#         mask: pd.Series = ~data.str.match(self.config["regex"]) & data.notna()
#         return mask
# df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
# regex_columns = ["ID"]
# for column in regex_columns:
#     # Pre procressing
#     df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
#     # Validation
#     validation_result = RegexValidation({"regex": "^[0-9]{8}$"}).validate(df_internal_suspended_list[column])
#     # Record result
#     df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} does not match regex"}))
#     # Log result
#     logger.info(f"[{column}] [Check regex] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
# df_internal_suspended_list

In [46]:
class DatetimeFormatValidation(ValidationStrategy):
    def __init__(self, config):
        super().__init__(config)

    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        mask: pd.Series = pd.to_datetime(data, errors="coerce", format=self.config["datetime_format"]).isna() & data.notna()
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
datetime_columns = ["Effective from"]
for column in datetime_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    validation_result = DatetimeFormatValidation({"datetime_format": "%d/%m/%Y"}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not datetime format '%d/%m/%Y'"}))
    # Log result
    logger.info(f"[{column}] [Check datetime] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
df_internal_suspended_list

2026-02-20 10:48:21.932 | INFO     | __main__:<module>:19 - [Effective from] [Check datetime] 7/21 records invalid. Index example: [3, 5, 7, 10, 11]


,ID,Debtor type,NRIC/FIN,UEN,Effective from,Effective to,Bad debt write-off amount,Suspended reason,Remarks,Status,Created On,Created By,Modified On,Modified By,validation_result
0,IS00000,Individual,T2213914F,NaN,25/07/2024,22/11/2024,abc,NaN,NaN,Active,abc,Rian,abc,Rian,{}
1,IS00001,Company,NaN,RianCompanyUEN2,31/12/1899,31/12/1899,123a,NaN,NaN,Inactive,123,Rian,123,Rian,{}
2,255vpPNwTliRDeOmMgTUfeexRZsSoOZlWBxfrPKtwyXPRd...,Individual,F2327892KZZ,NaN,NaN,3000-01-01 00:00:00,NaN,,NaN,active,01/09/2024 123 abc $%^,Rian,01/09/2024 123 abc $%^,Rian,{}
3,256AvpPNwTliRDeOmMgTUfeexRZsSoOZlWBxfrPKtwyXPR...,Company,NaN,RianCompanyUEN2,31/12/2999,31/12/2999,NaN,RiSuspensionReason_4,NaN,inactive,#$%,Rian,#$%,Rian,{Effective from is not datetime format '%d/%m/...
4,Abc,Individual,NaN,NaN,NaN,1900-01-06 23:57:25,NaN,RiSuspensionReason_5,NaN,ACTIVE,30-11-2024,Rian,30-11-2024,Rian,{}
5,NaN,Company,NaN,255vpPNwTliRDeOmMgTUfeexRZsSoOZlWBxfrPKtwyXPRd...,@,@,1,RiSuspensionReason_6,NaN,INACTIVE,30.09.2024 00:00,Rian,30.09.2024,Rian,{Effective from is not datetime format '%d/%m/...
6,$%^,Individual,,NaN,NaN,2024-05-08 00:00:00,1,RiSuspensionReason_7,NaN,Active,NaN,Rian,NaN,Rian,{}
7,IS00007,Company,NaN,256AvpPNwTliRDeOmMgTUfeexRZsSoOZlWBxfrPKtwyXPR...,01/10/2024 06:50:52,29/01/2025 06:50:52,1,RiSuspensionReason_88,NaN,Inactive,NaN,Rian,NaN,Rian,{Effective from is not datetime format '%d/%m/...
8,IS00008,Individual,T2127144J,NaN,NaN,NaN,NaN,RISUSPENSIONREASON_9,NaN,Active,,NaN,,Rian,{}
9,IS00009,Company,NaN,NaN,25/04/2024,25/04/2024,NaN,risuspensionreason_10,NaN,Inactiveeee,12/12/2002 00:00:00,NaN,12/12/2002 00:00:00,Rian,{}


In [ ]:
class InRangeDateValidation(ValidationStrategy):

    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        min_date = pd.to_datetime(self.config["min_date"], errors="coerce", format=self.config["datetime_format"])
        max_date = pd.to_datetime(self.config["max_date"], errors="coerce", format=self.config["datetime_format"])
        data = pd.to_datetime(data, errors="coerce", format=self.config["datetime_format"])

        mask = (
            data.notna() &
            ((data < min_date) | (data > max_date))
        )
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
datetime_columns = ["Effective from"]
for column in datetime_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan).replace("nan", np.nan)
    # Validation
    validation_result = InRangeDateValidation({"datetime_format": "%d/%m/%Y", "min_date": "01/01/2024", "max_date": "31/12/2999"}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not in range date"}))
    # Log result
    logger.info(f"[{column}] [Check in range date] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}. All invalid values: {df_internal_suspended_list.loc[validation_result, column]}")
df_internal_suspended_list

In [ ]:
class InRangeNumberValidation(ValidationStrategy):
    def validate(self, data) -> pd.Series:
        self.is_empty(data) # should be before any other validation. Mean that check df is not empty first, if empty, log warning and skip validation
        data = pd.to_numeric(data, errors="coerce")
        mask = (
            data.notna() &
            ((data < self.config["min_value"]) | (data > self.config["max_value"]))
        )
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
number_columns = ["Bad debt write-off amount"]
for column in number_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    # df_internal_suspended_list[column] = pd.to_numeric(df_internal_suspended_list[column], errors="coerce")
    validation_result = InRangeNumberValidation({"min_value": 0, "max_value": 100}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not in range number"}))
    # Log result
    logger.info(f"[{column}] [Check in range number] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}. All invalid values: {df_internal_suspended_list.loc[validation_result, column]}")
df_internal_suspended_list


In [ ]:
class InRangeStringLengthValidation(ValidationStrategy):
    def validate(self, data) -> pd.Series:
        self.is_empty(data) # should be before any other validation. Mean that check df is not empty first, if empty, log warning and skip validation
        mask = (
            data.notna() &
            ((data.str.len() < self.config["min_length"]) | (data.str.len() > self.config["max_length"]))
        )
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
string_columns = ["ID"]
for column in string_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    validation_result = InRangeStringLengthValidation({"min_length": 1, "max_length": 255}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not in range string length"}))
    # Log result
    logger.info(f"[{column}] [Check in range string length] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}. All invalid values: {df_internal_suspended_list.loc[validation_result, column]}")
df_internal_suspended_list

In [ ]:
class NumericTypeValidation(ValidationStrategy):
    def validate(self, data) -> pd.Series:
        self.is_empty(data) # should be before any other validation. Mean that check df is not empty first, if empty, log warning and skip validation
        mask = data.notna() & (pd.to_numeric(data, errors="coerce").isna())
        return mask
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
number_columns = ["Bad debt write-off amount"]
for column in number_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    # df_internal_suspended_list[column] = pd.to_numeric(df_internal_suspended_list[column], errors="coerce")
    validation_result = NumericTypeValidation({}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is not integer type"}))
    # Log result
    logger.info(f"[{column}] [Check numeric type] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}. All invalid values: {df_internal_suspended_list.loc[validation_result, column]}")
df_internal_suspended_list

In [ ]:
class IntergerTypeValidation(ValidationStrategy):
    INT_TYPE = (int, np.integer)
    def validate(self, data) -> pd.Series:
        self.is_empty(data) # should be before any other validation. Mean that check df is not empty first, if empty, log warning and skip validation
        mask = data.notna() & (pd.to_numeric(data, errors="coerce").isna() | data.map(lambda x: any([i for i in str(x) if i in [".", "-"]])))
        return mask

df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
number_columns = ["Bad debt write-off amount"]
for column in number_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip().replace("", np.nan)
    # Validation
    # df_internal_suspended_list[column] = pd.to_numeric(df_internal_suspended_list[column], errors="coerce")
    validation_result = IntergerTypeValidation({}).validate(df_internal_suspended_list[column])
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"]
    # Log result
    logger.info(f"[{column}] [Check integer type] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid. Index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}. All invalid values: {df_internal_suspended_list.loc[validation_result, column]}")
df_internal_suspended_list

In [ ]:
pd.to_numeric(df_internal_suspended_list.loc[15, "Bad debt write-off amount"]) | isinstance(df_internal_suspended_list.loc[15, "Bad debt write-off amount"], int)

In [47]:
VALIDATION_RULES = {
    "not_null": MandatoryValidation,
    "date_format": DatetimeFormatValidation,
    "date_range": InRangeDateValidation,
    "numeric_range": InRangeNumberValidation,
    "string_range": InRangeStringLengthValidation,
    "value_list": ValueListValidation,
    "int_type": IntergerTypeValidation,
    "numeric_type": NumericTypeValidation
}

In [ ]:
class InnerReferenceValidation(ValidationStrategy):
    def __init__(self, config):
        super().__init__(config)

    def validate(self, data) -> pd.Series:
        self.is_empty(data)
        if not self.config["ref_query"]:
            raise ValueError("Reference query not found")

        notna_mask = self.notna(data)

        for ref_info in self.config["ref_info"]:
            ref_filter_condition = ref_info.get("ref_filter_condition")
            ref_rules = ref_info.get("ref_rules")
            if not ref_filter_condition:
                raise ValueError("Reference filter condition not found")
            if not ref_rules:
                raise ValueError("Reference rules not found")

            mask = notna_mask & data.query(ref_filter_condition)

            for rule in ref_rules.items():
                validation_rule = VALIDATION_RULES.get(rule.get("type"))
                validation_config = rule.get("config")
                if not validation_rule:
                    raise ValueError(f"Invalid validation rule: {rule}")
                mask = mask & validation_rule({}).validate(data[mask])

        return mask

